In [ ]:
!nvidia-smi

In [ ]:
from diffusers import StableDiffusionXLPipeline
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from PIL import ImageFont
import torch
import json
import glob
import os

from imageGenerateUtils import get_image, add_caption

In [ ]:
stable_pretrained_model_link_or_path = "plantMilkModelSuite_flax.safetensors"

title_font = ImageFont.truetype("ipagp.ttf", 27)
paragraph_font = ImageFont.truetype("ipagp.ttf", 15)
caption_font = ImageFont.truetype("ipagp.ttf", 12)

input_dir = "generated_texts"
output_dir = "endemic"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
pipe = StableDiffusionXLPipeline.from_single_file(
    pretrained_model_link_or_path=stable_pretrained_model_link_or_path,
    torch_dtype=torch.float16
).to(device="cuda")

In [ ]:
json_files = sorted(glob.glob(os.path.join(input_dir, "*.json")))
print(f"Found {len(json_files)} text files")

In [ ]:
finalImages = []
for json_path in tqdm(json_files):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    name = data["name"]
    description = data["description"]
    prompt = data["prompt"]
    scientific_name = data["scientific_name"]

    background = get_image(prompt.strip(), pipe)
    finalImage = add_caption(name, description, scientific_name, background, title_font, paragraph_font, caption_font)

    basename = os.path.splitext(os.path.basename(json_path))[0]
    finalImage.save(os.path.join(output_dir, f"{basename}.png"))
    finalImages.append(finalImage)

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(40,24))
plt.subplots_adjust(wspace=0.1, hspace=0.1)
for ax, img in tqdm(zip(axes.flatten(), finalImages)):
    ax.imshow(img)
    ax.axis('off')